In [ ]:
# url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'
# resp = requests.get(url)


In [ ]:
# import io
# import zipfile
# import requests
# import frontmatter

In [ ]:
import io
import zipfile
import requests
import frontmatter


def read_repo_data(repo_owner, repo_name, branch='main', prefix=''):
    """Download a GitHub repo and parse its markdown files.

    Args:
        repo_owner: GitHub user or organization
        repo_name: Repository name
        branch: Branch to download
        prefix: Keep only files whose path starts with this value
                (the zip folder name is removed first)
    """
    url = f'https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/{branch}'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()

    repository_data = []

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            if file_info.is_dir():
                continue

            # Remove the "faq-main/" folder from the path
            relative_path = file_info.filename.split('/', 1)[1]

            if not relative_path.lower().endswith(('.md', '.mdx')):
                continue
            if not relative_path.startswith(prefix):
                continue

            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode('utf-8', errors='replace')
                data = frontmatter.loads(content).to_dict()
                data['filename'] = relative_path
                repository_data.append(data)
            except Exception as e:
                print(f"Error processing {relative_path}: {e}")

    return repository_data

In [ ]:
# faq_data = read_repo_data('DataTalksClub', 'faq', prefix='_questions/')
# print(faq_data[0])
evidently_docs = read_repo_data('evidentlyai', 'docs')
print(evidently_docs[45])


In [ ]:
# Just chunking by hand

import re

text = evidently_docs[45]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())
print(paragraphs)

# def sliding_window(seq, size, step):
#     if size <= 0 or step <= 0:
#         raise ValueError("size and step must be positive")

#     n = len(seq)
#     result = []
#     for i in range(0, n, step):
#         chunk = seq[i:i+size]
#         result.append({'start': i, 'chunk': chunk})
#         if i + size >= n:
#             break

#     return result

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections    

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)
        

In [65]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)


def llm(prompt, model='openai/gpt-oss-20b', max_output_tokens=8000):
    response = openai_client.responses.create(
        model=model,
        input=[{"role": "user", "content": prompt}],
        max_output_tokens=max_output_tokens,
    )
    if response.status == 'incomplete':
        raise RuntimeError(f"Incomplete reply: {response.incomplete_details}")
    return response.output_text

In [66]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

===SECTION===

## Another Section Name

Another section content

===SECTION===

Separate sections with a line that contains only: ===SECTION===
Never split inside a table or a code block.
""".strip()

In [68]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('===SECTION===')
    sections = [s.strip() for s in sections if len(s.strip()) >= 50]
    return sections

In [69]:
from tqdm.auto import tqdm

docs_with_text = [d for d in evidently_docs if d['content'].strip()]

evidently_chunks = []

for doc in tqdm(docs_with_text[:3]):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

  0%|          | 0/3 [00:00<?, ?it/s]

In [70]:
# Sanity checks

import statistics

print(len(evidently_chunks), "chunks")
lengths = [len(c['section']) for c in evidently_chunks]
print("min", min(lengths), "median", statistics.median(lengths), "max", max(lengths))

tiny = [c for c in evidently_chunks if len(c['section'].strip()) < 50]
print(len(tiny), "tiny chunks")

for doc in docs_with_text[:3]:
    original = len(doc['content'])
    joined = sum(len(c['section']) for c in evidently_chunks if c['filename'] == doc['filename'])
    print(doc['filename'], original, joined)

35 chunks
min 135 median 284 max 2197
0 tiny chunks
api-reference/introduction.mdx 783 768
changelog/changelog.mdx 5888 6754
docs/library/data_definition.mdx 11046 10119


In [ ]:
# Full run

import json
import time

evidently_chunks = []
failed = []

for doc in tqdm(docs_with_text):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    try:
        sections = intelligent_chunking(doc_content)
    except Exception as e:
        failed.append((doc['filename'], str(e)))
        time.sleep(10)
        continue

    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

    time.sleep(2)

with open('evidently_chunks.json', 'w') as f:
    json.dump(evidently_chunks, f)

print(len(evidently_chunks), "chunks;", len(failed), "failed")
print(failed)

  0%|          | 0/91 [00:00<?, ?it/s]